# 서울시 집계구별 재무 특성 기반 신용대출 보유 여부 예측

신용카드 이용 패턴, 소득 구성(급여소득자/자영업자 비율) 등 재무 특성을 바탕으로 신용대출 보유 여부(다수 보유 집계구인지)를 예측하는 분류 모델을 만듭니다.

In [ ]:
import warnings
warnings.filterwarnings(action='ignore')

import numpy as np
import pandas as pd

df = pd.read_csv('data/소비활동지수를_위한_금융통계.csv', encoding='cp949')
df.head()

## 데이터 선택 및 타겟 재정의

원래는 '성별'을 타겟으로 시도했으나, 지표만으로는 성별이 잘 구분되지 않아, 비즈니스적으로 더 의미 있는 타겟으로 재구성했습니다.

**새 타겟**: `신용대출_보유자_수(CRDT_LN_CNT)`를 중앙값 기준으로 이진화한 `대출보유_상위50%` (해당 집계구의 신용대출 보유자 비중이 상대적으로 높은지 여부)

**데이터 누수 방지**: 타겟을 만드는 데 쓰인 `신용대출_보유자_수`와, 그와 직접 연동되는 `평균_신용대출_대출잔액`(보유자가 없으면 0에 가까워 사실상 타겟을 그대로 노출)은 피처에서 제외했습니다.

In [ ]:
feature_cols = [
    '급여소득자_수(MDL4_CNT)',
    '자영업자_수(MDL5_CNT)',
    '기타(무직,주부,학생등)수(MDL9_CNT)',
    '주택담보대출_보유자_수(HOUS_LN_CNT)',
    '평균_주택담보대출_대출잔액(HOUS_LN_BAL)',
    '평균_신용카드_일시불이용금액(3개월)(SIN_FUL_M3_AMT)',
    '평균_신용카드_할부이용금액(3개월)(SIN_INSTL_M3_AMT)',
    '평균_신용카드_해외이용금액(3개월)(SIN_ABRD_M3_AMT)',
]
target_source_col = '신용대출_보유자_수(CRDT_LN_CNT)'

df_subset = df[feature_cols + [target_source_col]].copy()
df_subset.describe()

In [ ]:
print(df_subset.isnull().sum()) # 확인 결과 결측치는 없음

In [ ]:
# 상위 50% 기준 이진 타겟 생성 (해당 집계구의 신용대출 보유자 수가 50%보다 높은지)
median_val = df_subset[target_source_col].median()
df_subset['대출보유_상위50%'] = (df_subset[target_source_col] > median_val).astype(int)
print(df_subset['대출보유_상위50%'].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

X = df_subset[feature_cols]
Y = df_subset['대출보유_상위50%']

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, stratify=Y, random_state=42)

X_train, X_val, Y_train, Y_val = train_test_split(
    X_train, Y_train, test_size=0.1, stratify=Y_train, random_state=42)

print(X_train.shape, X_test.shape)

## 모델 학습 및 비교

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

models = {
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=200, eval_metric='logloss', random_state=42)
}

results = {}
for name, model in models.items():
    pred = model.fit(X_train, Y_train).predict(X_val)
    acc = accuracy_score(Y_val, pred)
    results[name] = acc
    print(f'[{name}] 정확도: {acc:.3f}\n{classification_report(Y_val, pred)}\n')

### 하이퍼파라미터 튜닝 (Dicision Tree)

In [ ]:
from scipy.stats import randint, uniform
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

# 1. DecisionTree 탐색할 하이퍼파라미터 공간 정의

params = {'min_impurity_decrease': uniform(0.001, 0.001),
         'max_depth': randint(20, 50),
         'min_samples_split': randint(2, 25),
         'min_samples_leaf': randint(1, 25),
         }

# 2. RandomizedSearchCV 설정 (Validation을 위해 cv=5 적용)
gs = RandomizedSearchCV(
    DecisionTreeClassifier(random_state=42),
    params,
    n_iter=100,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    random_state=42,
)

# 3. Train 데이터로 튜닝 수행
gs.fit(X_train, Y_train)

# 4. 최적 파라미터 출력
print('[DecisionTree 최적 하이퍼파라미터]')
print(gs.best_params_)

# 5. Validation 데이터로 튜닝된 성능 최종 확인
best_gs = gs.best_estimator_
val_pred_gs = best_gs.predict(X_val)

print('\n[Validation Set 성능]')
print(f'Validation Accuracy: {accuracy_score(Y_val, val_pred_gs):.3f}')
print(classification_report(Y_val, val_pred_gs))

### 하이퍼파라미터 튜닝 (XGBoost)

In [ ]:
from scipy.stats import uniform, randint
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

xgb_params = {
    'max_depth': randint(3, 8),
    'learning_rate': uniform(0.01, 0.2),
    'n_estimators': randint(100, 300),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4)
}

xgb_gs = RandomizedSearchCV(XGBClassifier(eval_metric='logloss', random_state=42),
    xgb_params,
    n_iter=50,
    scoring='f1',       # 클래스 불균형이 있으니 f1 스코어 기준이 유리합니다!
    cv=3,
    n_jobs=-1,
    random_state=42
)

xgb_gs.fit(X_train, Y_train)
print("XGBoost 최적 파라미터:", xgb_gs.best_params_)

best_xgb_gs = xgb_gs.best_estimator_
val_pred_xgb_gs = best_xgb_gs.predict(X_val)

print('\n[Validation Set 성능]')
print(f'Validation Accuracy: {accuracy_score(Y_val, val_pred_xgb_gs):.3f}')
print(classification_report(Y_val, val_pred_xgb_gs))

### 예측 결과 확인 (실제값 vs 예측값)

In [ ]:
result_df = X_test.copy()
result_df['실제값'] = Y_test.values
result_df['예측값'] = best_xgb_gs.predict(X_test)
result_df['정답여부'] = result_df['실제값'] == result_df['예측값']

print("예측 결과 샘플:")
print(result_df.head(10))

print("\n틀린 케이스:")
print(result_df[result_df['정답여부'] == False].head(5))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

cm = confusion_matrix(Y_test, best_xgb_gs.predict(X_test))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['하위50%', '상위50%'])
disp.plot(cmap='Blues')
plt.title('XGBoost Confusion Matrix')
plt.show()

In [ ]:
# 변수 중요도
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': best_xgb_gs.feature_importances_
}).sort_values('importance', ascending=False)
print(importance)

In [ ]:
from sklearn.metrics import roc_auc_score

proba = best_xgb_gs.predict_proba(X_test)[:, 1]
print("Test AUC:", roc_auc_score(Y_test, proba))

In [ ]:
# 최종 선택된 XGBoost 모델(best_xgb_gs)로 테스트 데이터(Test Set) 최종 평가
final_pred = best_xgb_gs.predict(X_test)

print("[최종 Test Set 평가 결과 - XGBoost]")
print(classification_report(Y_test, final_pred))

## 결론 및 한계

**진행 과정 요약**
- 처음에는 '성별'을 타겟으로 예측을 시도했으나, 정확도가 55~58%로 랜덤 수준(50%)과 큰 차이가 없어 재무 데이터만으로는 성별이 잘 구분되지 않는다는 것을 확인했습니다.
- 이를 바탕으로, 데이터가 실제로 담고 있는 신용/대출 관련 정보를 활용하는 쪽으로 타겟을 '신용대출 보유자 비중'으로 재정의했습니다.

**주요 발견**
- 변수 중요도 상위 항목은 급여소득자 수, 기타(무직·주부·학생 등) 수, 
  신용카드 일시불 이용액 순으로 나타났으나, 항목 간 중요도 차이는 크지 않았습니다.
- 최종 테스트 세트 성능은 accuracy 0.53, f1-score 약 0.49~0.56 수준으로,
  두 클래스가 균형(54:46)을 이루도록 타겟을 재정의했음에도 랜덤 수준(50%)을 
  크게 벗어나지 못했습니다. 이는 주어진 8개 재무 피처만으로는 신용대출 보유 
  비중(상위/하위 50%)을 안정적으로 구분하기 어렵다는 것을 시사합니다.
- 최종 모델의 Test AUC는 0.49로, 완전 무작위 수준(0.5)과 사실상 동일했습니다.
  이는 accuracy(0.53)가 baseline(다수 클래스 비율, 0.54)과 비슷했던 것과
  일치하는 결과로, 급여소득자 수·자영업자 수·신용카드 이용 패턴 등 
  현재 확보한 8개 재무 피처만으로는 집계구의 신용대출 보유 비중
  (상위/하위 50%)을 구분할 수 있는 정보가 담겨 있지 않다는 것을 
  통계적으로 확인했습니다.

**한계**
- 집계구 단위로 집계된 데이터라 개인 단위 신용 이력, 신용점수, 
  기존 부채 상태 등 실제 대출 심사에 쓰이는 핵심 정보가 빠져 있어,
  간접적인 소비/소득 패턴만으로는 예측이 어려웠을 가능성이 큽니다.
- 표본 수(500개)의 한계보다는, AUC가 0.5에 근접한 것으로 볼 때
  '표본 부족'보다는 '피처 자체의 정보 부족'이 더 근본적인 원인으로 보입니다.
- Validation set이 40개로 작아, 모델 간 성능 비교(Decision Tree/RandomForest/
  XGBoost) 결과의 변동폭이 컸을 가능성이 있습니다.

**배운 점**
- 타겟 변수 선택이 모델 성능과 프로젝트의 설득력에 미치는 영향이 크다는 것을 확인했습니다. 단순히 예측 가능한 변수를 고르기보다, 데이터가 실제로 담고 있는 정보와 비즈니스 맥락에 맞는 타겟을 고르는 것이 중요하다는 것을 배웠습니다.